# Сравнение: FEM → coarse Galerkin → GMsFEM → neural operators

Фиксируем одну и ту же задачу для каждого held-out примера: DG0 A,n, k, направление и исходную тонкую FEM-сетку. Метрика всех методов:
\[
\|u_{method}^s-u_{FEM}^s\|_M/\|u_{FEM}^s\|_M.
\]
Fine FEM — reference, а не точное аналитическое решение. Его почти нулевая ошибка к собственным меткам не означает превосходство по точности. Coarse baseline — Q1 traces, интерполированные в fine P1 и решённые Galerkin-проекцией; это не самостоятельная coarse P1-сетка. GMsFEM использует существующие snapshots Type 1/2 и спектральные пространства `gmsfem.offline`.

Основная paired-таблица содержит по одному направлению **трёх** test-частот для всех методов. Полная neural-таблица по всем 1280 test-примерам показывается отдельно. Повторяющиеся направления одной частоты статистически зависимы. Числа параметров сетей и FEM DOF не являются одной и той же величиной.

**Статичная задача:** геометрия и исходная среда №0 фиксированы; A,n и DG0-материалы одинаковы во всех примерах. Меняются k и θ. Один medium_id=0, разбиение по frequency_id: 670/170/160 частот (5360/1360/1280 полей). 1000 частот × 8 направлений = 8000 полей на полосу. Этот эксперимент проверяет интерполяцию параметров волны на одной среде.

**Профиль:** выберите `POD_METHOD="full"` или `"randomized"` в первой ячейке. Данные общие; пути результатов разделены. Сохранённые outputs относятся к randomized. Это выбор алгоритма POD, а не отдельные версии Chebyshev/FNO.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'neural_operator').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from neural_operator import settings as full_settings
from neural_operator.randomized import settings as randomized_settings
POD_METHOD = "randomized"  # "full" uses separate runs.
if POD_METHOD not in {"full", "randomized"}:
    raise ValueError("POD_METHOD must be full or randomized")
settings = randomized_settings if POD_METHOD == "randomized" else full_settings
BASE, N_FREQUENCIES, N_DIRECTIONS, DATASET_TAG = (settings.BASE, settings.N_FREQUENCIES,
    settings.N_DIRECTIONS, settings.DATASET_TAG)
LOW_DATA_DIR = settings.LOW_DATA_DIR
LOW_RUNS = settings.LOW_RUNS
SVD_RUNS, COMPARISON_RUNS = settings.SVD_RUNS, settings.COMPARISON_RUNS
DATA_DIR = LOW_DATA_DIR
RUNS = ROOT / 'neural_operator' / 'runs' / 'revision_2026-11-08' / 'static_low'
MODEL_RUNS = {
    'svd': RUNS / 'svd', 'chebyshev': RUNS / 'chebyshev',
    'fno': ROOT / 'neural_operator' / 'randomized' / 'runs' / 'low_frequency_static_1000x8' / 'fno',
}
import json
import numpy as np
import matplotlib.pyplot as plt
from neural_operator.datasets import load_dataset, write_dataset_info
from neural_operator.plotting import display_table, plot_prediction, plot_history


In [ ]:
from neural_operator.experiments import compare_methods
from neural_operator.diagnostics import write_csv, transfer_diagnostics
dataset = load_dataset(DATA_DIR, mmap=True)
OUT = RUNS / 'comparison'
METHODS = ['svd', 'chebyshev', 'fno']
run_dirs = {method: MODEL_RUNS[method] for method in METHODS}
paired, summaries, neural_full = compare_methods(dataset, run_dirs, OUT,
    n_groups=3, n_coarse=4, modes=(1, 2, 4), basis_types=(1, 2))
display_table(summaries)


## Полный test только для нейрооператоров

Времена warm inference включают кодирование входа, вызов модели и восстановление FEM-поля. Для GMsFEM отдельно сохранены assembly, построение базиса и полный online solve с реконструкцией. Базис строится один раз до максимального M серии; соответствующая стоимость указывается во всех строках серии. Для фиксированных A базис Type 1 можно переиспользовать между частотами; Type 2 зависит от k. Здесь offline cost измеряется при каждом построении в benchmark. Training и preprocessing сетей показаны отдельно и не скрыты в оценке ускорения.


In [ ]:
display_table(neural_full)
transfer_rows = transfer_diagnostics(dataset, dataset.split_indices()[2], resolution=32)
write_csv(OUT / 'transfer_test.csv', transfer_rows)
display_table(transfer_rows)


In [ ]:
visible = [row for row in summaries if row['method'] != 'Fine FEM reference']
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
labels = [r['method'] for r in visible]
y = np.arange(len(labels))
axes[0].barh(y, [r['mean'] for r in visible]); axes[0].set_yticks(y, labels)
axes[0].set(xlabel='Paired mean relative FEM L2')
axes[1].barh(y, [r['online_seconds_mean'] for r in visible]); axes[1].set_yticks(y, labels)
axes[1].set(xscale='log', xlabel='Warm online time, seconds; offline costs in table')
fig.savefig(OUT / 'paired_comparison.png', dpi=160)
plt.show()


Небольшой seed-фиксированный эксперимент не определяет универсальный рейтинг методов. На локальные GMsFEM пространства влияют число мод и локальные резонансы; на нейрооператоры — число сред, пространственный перенос и tuning budget. Статьи и точные отличия реализации: [docs/references.md](../docs/references.md).
